# 作业 3.1：HPGe $\gamma$ 能谱刻度

## 1. 方法

HPGe 能谱刻度从局部峰拟合开始。对每个刻度峰采用同一个

$$f(x)=s(x)+b(x)$$

模型：$s(x)$ 是 photopeak signal，$b(x)$ 是局部本底。孤立且近似对称的峰可用 Gaussian signal；本底先用线性函数，低能侧形成明显 step 时再加入 `erfc` 项。模型是否足够由拟合残差判断。

<img src="../calibration_method/peak_model_components.png" alt="photopeak signal, local background, total fit and residual" style="max-width:75%;" />

同一次拟合给出三类刻度量：

- 峰中心 $ch_i$ 用于能量刻度。先拟合 $E=a_0+a_1ch$，只有残差显示系统曲率时才加入 $a_2ch^2$。
- $\sigma_{ch}$ 先换算为 $\sigma_E=|dE/dch|\sigma_{ch}$，再计算 $FWHM=2.355\sigma_E$。
- 峰面积取信号分量的积分。对等宽 histogram 中的 Gaussian peak，

  $$N_{\mathrm{peak}}=\frac{A\sigma\sqrt{2\pi}}{w},$$

  其中 $A$ 是 Gaussian height，$w$ 是 bin width。这里不把 covariance 作为峰拟合的输入；峰拟合完成后，用 ROOT 返回的参数 covariance matrix 中的 $\operatorname{Cov}(A,\sigma)$ 传播峰面积误差。

<img src="../calibration_method/full_energy_peak_area.png" alt="full-energy peak area above the fitted local background" style="max-width:75%;" />

Full-energy peak efficiency 使用同一个 $N_{\mathrm{peak}}$：

$$
\varepsilon(E_\gamma)=
\frac{N_{\mathrm{peak}}}{A(t)P_\gamma t_{\mathrm{live}}}.
$$

$A(t)$ 是测量时活度，$P_\gamma$ 是每次衰变发射该 gamma ray 的概率。Sideband subtraction 只作为独立交叉检查，不与上述拟合本底混用。

## 2. 作业

### 2.1 数据与实验设置

使用 [gamma.root](gamma.root) 中的 `TH1F h0` 完成 Eurica 探测阵列的能量、峰宽和效率刻度。`h0` 是 $^{152}$Eu 与 $^{133}$Ba 标准源的合并能谱，横轴是尚未刻度的 pulse-height coordinate，范围为 0–2500，每个 bin 宽 0.2。数据已对每个 Euroball Cluster 的 7 个晶体进行 add-back，再合并 12 个 Cluster。

Eurica 由 12 个 Euroball Cluster 组成，每个 Cluster 含 7 个 HPGe 晶体，标定源距探测器约 22 cm。

![Eurica detector array](eurica.png)

数据采集于 2013 年 2 月 13 日。源的参考日期为 1998 年 1 月 1 日；参考活度为 $^{152}$Eu 40.9 kBq（5%）和 $^{133}$Ba 42.2 kBq（3%）。记录时长为 7442 s。效率计算中先把 7442 s 当作 live time；若它实际是 wall-clock time，则还需要 dead-time correction。

### 2.2 刻度线

$P_\gamma$ 是每次核衰变发出该 gamma ray 的概率，不是源活度；它既用于辅助判断谱线的相对显著程度，也用于后面的效率计算。

| Nuclide | $E_\gamma$ (keV) | $P_\gamma$ (%) |
| --- | ---: | ---: |
| $^{133}$Ba | 80.9979 | 34.06 |
| $^{152}$Eu | 121.7817 | 28.41 |
| $^{152}$Eu | 244.6974 | 7.55 |
| $^{133}$Ba | 276.3989 | 7.164 |
| $^{133}$Ba | 302.8508 | 18.33 |
| $^{152}$Eu | 344.2785 | 26.59 |
| $^{133}$Ba | 356.0129 | 62.05 |
| $^{152}$Eu | 778.9045 | 12.93 |
| $^{152}$Eu | 867.378 | 4.23 |
| $^{152}$Eu | 964.079 | 14.51 |
| $^{152}$Eu | 1112.076 | 13.67 |
| $^{152}$Eu | 1408.013 | 20.87 |

能量与 pulse-height coordinate 近似满足线性关系。先在 log scale 下寻找较显著的峰候选；对同一核素，可用 $P_\gamma$ 的相对大小辅助判断，对两个不同核素则还要计入各自的 $A(t)$。选择两个相隔较远且指认可靠的峰估计初步线性关系，再用它预测并核对其他刻度线的位置。观测峰强还随 full-energy peak efficiency 改变，因此 $A(t)P_\gamma$ 只用于初步定位，不能直接当作峰面积之比。

计算活度时可采用 $T_{1/2}(^{152}\mathrm{Eu})=13.517$ y、$T_{1/2}(^{133}\mathrm{Ba})=3849.3$ d。

### 2.3 作业要求

#### 能量刻度

1. 用 log scale 查看完整的 `h0`，按照线性关系和相对发射强度初步定位刻度峰。
2. 对各刻度峰进行局部拟合。峰与邻近结构重叠时应调整拟合区间，不能让本底函数吸收另一个峰。逐峰检查拟合残差。
3. 将峰中心和参考能量写入 `TGraphErrors`，峰中心误差作为横坐标误差。比较一次和二次刻度函数，并画出 $\Delta E=E_{\mathrm{ref}}-E_{\mathrm{cal}}$ 残差。
4. 用残差支持的刻度关系生成能量谱，并检查变换前后总计数是否守恒。

#### 峰宽与能量分辨率

由各峰的 $\sigma_{ch}$ 计算 FWHM，画出 FWHM–$E_\gamma$ 曲线，拟合

$$FWHM(E)=\sqrt{A+BE+CE^2},$$

并给出 $FWHM_{\mathrm{data}}-FWHM_{\mathrm{fit}}$ 残差。根据残差判断是否需要全部三项。

#### Full-energy peak efficiency（选做）

1. 从峰形拟合中的 signal 分量计算 $N_{\mathrm{peak}}$，并用拟合返回的参数 covariance matrix 传播其统计误差。
2. 将两个源的活度修正到测量日期，计算各条谱线的 apparent full-energy peak efficiency。
3. 在 log–log 坐标上拟合 efficiency–energy 曲线，并给出相对残差。
4. 说明 dead time、true-coincidence summing、源几何和自吸收修正对绝对效率的影响。

### 2.4 实例代码：两个典型峰

下面只演示两个局部峰：一个使用 Gaussian + linear background，另一个加入低能侧 `erfc` step。代码给出峰中心、$\sigma$、Gaussian signal 面积及其误差，并画出相应残差；其余刻度峰仍需按照作业要求自行定位和拟合。

<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code language:</span>
  <button type="button" data-code-language="python" aria-pressed="true">Python / PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:none; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
.pyroot-code-marker + .highlight {
  margin:.5rem 0 1rem;
  border:1px solid #d5d5d5;
  border-radius:2px;
  background:#f7f7f7;
}
.pyroot-code-marker + .highlight pre { margin:0; padding:.75rem 1rem; overflow-x:auto; }
.pyroot-code-cell[hidden],
.jp-CodeCell .jp-Cell-inputWrapper[hidden] { display:none !important; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); })
    .filter(Boolean);
  pythonCells.forEach(function (cell) { cell.classList.add("pyroot-code-cell"); });
  const cppInputs = document.querySelectorAll(".jp-CodeCell .jp-Cell-inputWrapper");

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppInputs.forEach(function (input) { input.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  document.querySelector(".code-language-switch").style.display = "flex";
  selectLanguage("python");
});
</script>

<div class="pyroot-code-marker"></div>

```python
import math
from array import array
import ROOT

ROOT.gStyle.SetOptStat(0)

input_file = ROOT.TFile.Open("gamma.root", "READ")
h0 = input_file.Get("h0")

c_spectrum = ROOT.TCanvas("c_spectrum_py", "h0", 850, 480)
c_spectrum.SetLogy()
h0.SetTitle("^{152}Eu + ^{133}Ba spectrum;pulse-height coordinate;counts / bin")
h0.GetXaxis().SetRangeUser(40, 1300)
h0.SetMinimum(0.5)
h0.Draw("hist")
c_spectrum.Draw()
```

In [1]:
#include "TCanvas.h"
#include "TFile.h"
#include "TF1.h"
#include "TFitResultPtr.h"
#include "TGraph.h"
#include "TH1.h"
#include "TLine.h"
#include "TMath.h"
#include "TStyle.h"
#include <algorithm>
#include <cmath>
#include <iomanip>
#include <iostream>
#include <string>
#include <vector>

gStyle->SetOptStat(0);
auto inputFile = TFile::Open("gamma.root", "READ");
auto h0 = dynamic_cast<TH1*>(inputFile->Get("h0"));

auto cSpectrum = new TCanvas("cSpectrum", "h0", 850, 480);
cSpectrum->SetLogy();
h0->SetTitle("^{152}Eu + ^{133}Ba spectrum;pulse-height coordinate;counts / bin");
h0->GetXaxis()->SetRangeUser(40, 1300);
h0->SetMinimum(0.5);
h0->Draw("hist");
cSpectrum->Draw();

<div class="pyroot-code-marker"></div>

```python
def fit_peak(hist, name, guess, half_width, use_step):
    xmin, xmax = guess - half_width, guess + half_width
    left = hist.GetBinContent(hist.FindBin(xmin))
    right = hist.GetBinContent(hist.FindBin(xmax))
    slope = (right - left) / (xmax - xmin)
    intercept = left - slope * xmin
    height = max(hist.GetBinContent(hist.FindBin(guess))
                 - intercept - slope * guess, 1.0)

    formula = "gaus(0)+pol1(3)"
    if use_step:
        formula += "+[5]*0.5*TMath::Erfc((x-[1])/(sqrt(2)*[2]))"
    function = ROOT.TF1(name, formula, xmin, xmax)
    function.SetParameters(height, guess, 0.8, intercept, slope)
    function.SetParLimits(0, 0.0, 10.0 * height)
    function.SetParLimits(1, guess - 2.0, guess + 2.0)
    function.SetParLimits(2, 0.2, 3.0)
    if use_step:
        function.SetParameter(5, max(left - right, 0.0))
        function.SetParLimits(5, 0.0, height)

    # L: binned Poisson likelihood; I: integrate over each bin;
    # R: use the TF1 range; S: return TFitResult; Q/N: quiet/no auto-draw.
    result = hist.Fit(function, "LIRSQN")

    amplitude = function.GetParameter(0)
    sigma = abs(function.GetParameter(2))
    factor = math.sqrt(2.0 * math.pi) / hist.GetBinWidth(1)
    area = amplitude * sigma * factor

    # The area error includes the covariance between height and sigma.
    covariance = result.GetCovarianceMatrix()
    d_area_d_amplitude = sigma * factor
    d_area_d_sigma = amplitude * factor
    area_variance = (
        d_area_d_amplitude**2 * covariance[0][0]
        + d_area_d_sigma**2 * covariance[2][2]
        + 2.0 * d_area_d_amplitude * d_area_d_sigma * covariance[0][2]
    )

    residual = ROOT.TGraph()
    for point, bin_number in enumerate(
        range(hist.FindBin(xmin), hist.FindBin(xmax) + 1)
    ):
        low = hist.GetBinLowEdge(bin_number)
        width = hist.GetBinWidth(bin_number)
        expected = function.Integral(low, low + width) / width
        observed = hist.GetBinContent(bin_number)
        residual.SetPoint(
            point, hist.GetBinCenter(bin_number),
            (observed - expected) / math.sqrt(expected)
        )

    return {
        "function": function,
        "status": int(result),
        "mean": function.GetParameter(1),
        "mean_error": function.GetParError(1),
        "sigma": sigma,
        "area": area,
        "area_error": math.sqrt(max(area_variance, 0.0)),
        "residual": residual,
        "range": (xmin, xmax),
    }

examples = [
    ("244.7 keV: Gaussian + linear", 239.1, 3.5, False),
    ("867.4 keV: Gaussian + linear + step", 762.7, 4.5, True),
]
fits = [
    fit_peak(h0, f"example_{i}_py", guess, width, step)
    for i, (_, guess, width, step) in enumerate(examples)
]

c_examples = ROOT.TCanvas("c_examples_py", "Example peak fits", 900, 700)
c_examples.Divide(2, 2)
views = []
zero_lines = []
for i, ((label, _, _, _), fit) in enumerate(zip(examples, fits)):
    xmin, xmax = fit["range"]
    c_examples.cd(i + 1)
    view = h0.Clone(f"example_view_{i}_py")
    view.GetXaxis().SetRangeUser(xmin, xmax)
    view.SetTitle(f"{label};pulse-height coordinate;counts / bin")
    view.Draw("E")
    fit["function"].SetLineColor(ROOT.kBlue + 1)
    fit["function"].Draw("same")
    views.append(view)

    c_examples.cd(i + 3)
    fit["residual"].SetTitle(
        f"{label};pulse-height coordinate;(n-#mu)/#sqrt{{#mu}}"
    )
    fit["residual"].SetMarkerStyle(20)
    fit["residual"].Draw("AP")
    line = ROOT.TLine(xmin, 0.0, xmax, 0.0)
    line.SetLineStyle(2)
    line.Draw()
    zero_lines.append(line)
c_examples.Draw()

print("peak                  mean +/- error       sigma       N_peak +/- error")
for (label, _, _, _), fit in zip(examples, fits):
    print(f"{label:35s} {fit['mean']:9.4f} +/- {fit['mean_error']:.4f}  "
          f"{fit['sigma']:7.4f}  {fit['area']:10.0f} +/- {fit['area_error']:.0f}  "
          f"status={fit['status']}")
```

In [2]:
{
struct PeakFit {
    TF1* function;
    TGraph* residual;
    int status;
    double mean;
    double meanError;
    double sigma;
    double area;
    double areaError;
    double xmin;
    double xmax;
};

auto fitPeak = [&](const std::string& name, double guess,
                   double halfWidth, bool useStep) -> PeakFit {
    const double xmin = guess - halfWidth;
    const double xmax = guess + halfWidth;
    const double left = h0->GetBinContent(h0->FindBin(xmin));
    const double right = h0->GetBinContent(h0->FindBin(xmax));
    const double slope = (right - left) / (xmax - xmin);
    const double intercept = left - slope * xmin;
    const double height = std::max(
        h0->GetBinContent(h0->FindBin(guess)) - intercept - slope * guess, 1.0);

    std::string formula = "gaus(0)+pol1(3)";
    if (useStep) {
        formula += "+[5]*0.5*TMath::Erfc((x-[1])/(sqrt(2)*[2]))";
    }
    auto function = new TF1(name.c_str(), formula.c_str(), xmin, xmax);
    function->SetParameters(height, guess, 0.8, intercept, slope);
    function->SetParLimits(0, 0.0, 10.0 * height);
    function->SetParLimits(1, guess - 2.0, guess + 2.0);
    function->SetParLimits(2, 0.2, 3.0);
    if (useStep) {
        function->SetParameter(5, std::max(left - right, 0.0));
        function->SetParLimits(5, 0.0, height);
    }

    // L/I/R/S/Q/N have the same meanings as in the PyROOT version.
    TFitResultPtr result = h0->Fit(function, "LIRSQN");
    const double amplitude = function->GetParameter(0);
    const double sigma = std::abs(function->GetParameter(2));
    const double factor = std::sqrt(2.0 * TMath::Pi()) / h0->GetBinWidth(1);
    const double area = amplitude * sigma * factor;

    // Propagate height, sigma and their covariance to the signal area.
    const auto covariance = result->GetCovarianceMatrix();
    const double dAreaDA = sigma * factor;
    const double dAreaDSigma = amplitude * factor;
    const double areaVariance =
        dAreaDA * dAreaDA * covariance(0, 0)
        + dAreaDSigma * dAreaDSigma * covariance(2, 2)
        + 2.0 * dAreaDA * dAreaDSigma * covariance(0, 2);

    auto residual = new TGraph();
    int point = 0;
    for (int bin = h0->FindBin(xmin); bin <= h0->FindBin(xmax); ++bin) {
        const double low = h0->GetBinLowEdge(bin);
        const double width = h0->GetBinWidth(bin);
        const double expected = function->Integral(low, low + width) / width;
        const double observed = h0->GetBinContent(bin);
        residual->SetPoint(point++, h0->GetBinCenter(bin),
                           (observed - expected) / std::sqrt(expected));
    }

    return {function, residual, static_cast<int>(result),
            function->GetParameter(1), function->GetParError(1), sigma,
            area, std::sqrt(std::max(areaVariance, 0.0)), xmin, xmax};
};

const std::vector<std::string> labels = {
    "244.7 keV: Gaussian + linear",
    "867.4 keV: Gaussian + linear + step"};
std::vector<PeakFit> fits;
fits.push_back(fitPeak("example244", 239.1, 3.5, false));
fits.push_back(fitPeak("example867", 762.7, 4.5, true));

auto cExamples = new TCanvas("cExamples", "Example peak fits", 900, 700);
cExamples->Divide(2, 2);
for (std::size_t i = 0; i < fits.size(); ++i) {
    cExamples->cd(i + 1);
    auto view = static_cast<TH1*>(h0->Clone(("exampleView" + std::to_string(i)).c_str()));
    view->GetXaxis()->SetRangeUser(fits[i].xmin, fits[i].xmax);
    view->SetTitle((labels[i] + ";pulse-height coordinate;counts / bin").c_str());
    view->Draw("E");
    fits[i].function->SetLineColor(kBlue + 1);
    fits[i].function->Draw("same");

    cExamples->cd(i + 3);
    fits[i].residual->SetTitle(
        (labels[i] + ";pulse-height coordinate;(n-#mu)/#sqrt{#mu}").c_str());
    fits[i].residual->SetMarkerStyle(20);
    fits[i].residual->Draw("AP");
    auto zero = new TLine(fits[i].xmin, 0.0, fits[i].xmax, 0.0);
    zero->SetLineStyle(2);
    zero->Draw();
}
cExamples->Draw();

std::cout << "peak                  mean +/- error       sigma       N_peak +/- error\n";
for (std::size_t i = 0; i < fits.size(); ++i) {
    std::cout << std::fixed << std::setprecision(4)
              << std::setw(35) << labels[i]
              << "  " << std::setw(9) << fits[i].mean
              << " +/- " << fits[i].meanError
              << "  " << std::setw(7) << fits[i].sigma
              << "  " << std::setprecision(0) << std::setw(10) << fits[i].area
              << " +/- " << fits[i].areaError
              << "  status=" << fits[i].status << "\n";
}
}

peak                  mean +/- error       sigma       N_peak +/- error
       244.7 keV: Gaussian + linear   239.1914 +/- 0.0008   0.6519     1429873 +/- 1756  status=0
867.4 keV: Gaussian + linear + step   762.6353 +/- 0.0022   0.8199      480499 +/- 889  status=0


### 2.5 结果参考

完成作业后，可用下面的结果检查刻度关系、曲线形状和残差结构。一次能量刻度给出

$$E_\gamma\;(\mathrm{keV})\approx-39.943+1.189802\,ch,$$

最大绝对刻度残差约为 0.10 keV；二次项没有改善本数据的最大残差。

<img src="reference_calibrated_spectrum.png" alt="calibrated gamma spectrum" style="max-width:78%;" />

<img src="reference_energy_calibration.png" alt="energy calibration and residuals" style="max-width:78%;" />

<img src="reference_fwhm.png" alt="FWHM versus energy and residuals" style="max-width:78%;" />

<img src="reference_efficiency.png" alt="apparent full-energy peak efficiency and residuals" style="max-width:78%;" />